In [ ]:
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd

from restaurant_visitor_eda.features import add_visitor_aggregations, build_features_advanced

2026-05-19 18:19:25.604 | INFO     | restaurant_visitor_eda.config:<module>:11 - PROJ_ROOT path is: /home/alxdrzd/alxdrzd/ds-projects/restaurant-visitor-eda


In [ ]:
from restaurant_visitor_eda.config import PROCESSED_DATA_DIR

df_base = pd.read_csv(PROCESSED_DATA_DIR / "train_baseline.csv", parse_dates=["visit_date"])

df_base = build_features_advanced(df_base)

val_cutoff = "2017-03-14"
train_mask = df_base["visit_date"] < val_cutoff
val_mask = df_base["visit_date"] >= val_cutoff

df_train_local = df_base[train_mask].copy()
df_val_local = df_base[val_mask].copy()

df_train_feat = add_visitor_aggregations(df_train_local, df_train=df_train_local)
df_val_feat = add_visitor_aggregations(df_val_local, df_train=df_train_local)

In [ ]:
categorical_features = [
    "air_store_id",
    "air_genre_name",
    "day_of_week",
    "month",
    "day_pattern",
    "prefecture",
    "district",
    "block",
]
numeric_features = [
    "latitude",
    "longitude",
    "doy_sin",
    "doy_cos",
    "dow_sin",
    "dow_cos",
    "median_visitors_dow",
    "mean_visitors_dow",
    "min_visitors_dow",
    "max_visitors_dow",
    "median_visitors_total",
    "gw_genre_geo_median",
    "median_reserve_visitors_dow",
    "walk_in_ratio",
]
binary_features = ["is_gw", "has_gw_history"]
features = categorical_features + numeric_features + binary_features

df_train_feat[categorical_features] = (
    df_train_feat[categorical_features].fillna("Unknown").astype(str)
)
df_val_feat[categorical_features] = df_val_feat[categorical_features].fillna("Unknown").astype(str)

X_train = df_train_feat[features]
y_train = np.log1p(df_train_feat["visitors"])
X_val = df_val_feat[features]
y_val = np.log1p(df_val_feat["visitors"])

model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    cat_features=categorical_features,
    random_seed=42,
    od_type="Iter",
    od_wait=100,
)

In [ ]:
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True, verbose=100)

print(f"\n[Hold-out] Best Validation RMSLE: {model.get_best_score()['validation']['RMSE']:.4f}")

0:	learn: 0.7821594	test: 0.8104244	best: 0.8104244 (0)	total: 141ms	remaining: 3m 31s
100:	learn: 0.5065259	test: 0.5494151	best: 0.5494151 (100)	total: 8.18s	remaining: 1m 53s
200:	learn: 0.4985395	test: 0.5514787	best: 0.5485532 (128)	total: 15.9s	remaining: 1m 42s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.548553205
bestIteration = 128

Shrink model to first 129 iterations.

[Hold-out] Best Validation RMSLE: 0.5486


In [ ]:
df_test_raw = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv")

df_test_raw[categorical_features] = df_test_raw[categorical_features].fillna("Unknown").astype(str)

X_test = X_test = df_test_raw[features]

preds_log = model.predict(X_test)

preds_real = np.expm1(preds_log)


df_test_raw["visit_date"] = pd.to_datetime(df_test_raw["visit_date"])

submission = pd.DataFrame(
    {
        "id": df_test_raw["air_store_id"]
        + "_"
        + df_test_raw["visit_date"].dt.strftime("%Y-%m-%d"),
        "visitors": preds_real,
    }
)

submission_path = "submission_catboost_custom.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,2.190376
1,air_08cb3c4ee6cd6a22_2017-04-23,16.950511
2,air_f8233ad00755c35c_2017-04-23,7.166549
3,air_234d3dbf7f3d5a50_2017-04-23,8.622923
4,air_a563896da3777078_2017-04-23,35.610607
